# 05 - Outliers, Business Zones, and Interpretability

Convert technical clusters into operations-friendly business zones.


## Definition
Explain the concept in one sentence and why it matters in delivery analytics.

## Theory
Describe the assumptions and algorithmic behavior at a practical level.

## Mathematical Intuition
Show the formula or geometry intuition behind the method.

## Visual Explanation
Use a chart/map to make the concept concrete.

## Code Explanation
Walk through what each code block does and what inputs/outputs mean.

## Interpretation of Results
Connect metrics and visuals to operational decisions.


In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

if (Path.cwd() / "src").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise RuntimeError("Could not locate project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering


In [2]:
from src.config import TRAIN_FILE_PATH
from src.data_loader import load_raw_data, load_and_clean_data, explain_dataset_fields


In [3]:
from src.features import build_clustering_features, select_features
from src.outlier_detection import detect_outliers
from src.clustering import run_algorithm
from src.business_zones import build_cluster_kpis, assign_business_zone_labels, attach_zone_labels

df = load_and_clean_data(TRAIN_FILE_PATH)
feat = build_clustering_features(df.copy())
X = select_features(feat)
geo = feat[["Restaurant_latitude", "Restaurant_longitude"]].to_numpy(dtype=float)

consensus, reports = detect_outliers(feat, feature_matrix=X)
outlier_summary = pd.DataFrame({"method": [r.name for r in reports], "outliers": [r.n_outliers for r in reports]})
display(outlier_summary)

mask = ~consensus.mask
feat_clean = feat.loc[mask].reset_index(drop=True)
X_clean = X[mask]
geo_clean = geo[mask]

result = run_algorithm("kmeans", X_clean, geo_coords=geo_clean)
kpis = build_cluster_kpis(feat_clean, result.labels)
labeled_kpis, rule_summary = assign_business_zone_labels(kpis)
labeled_rows = attach_zone_labels(feat_clean, result.labels, labeled_kpis)

display(labeled_kpis)
display(labeled_rows[["cluster", "zone_label", "zone_reason"]].head())
print(rule_summary)


,method,outliers
0,coordinate_bounds,0
1,gps_error,0
2,iqr,231
3,mahalanobis,2171
4,isolation_forest,1201
5,lof,1201
6,dbscan_noise,1296


,cluster,orders,avg_distance_km,avg_pickup_lag_min,centroid_lat,centroid_lon,outlier_pct,coverage_pct,density_index,centrality_km,zone_label,zone_reason
0,1,11527,9.778437,9.996530,13.180887,77.915037,0.0,29.937149,1178.818238,639.289431,High Demand Zone,High order volume with strong density index.
1,3,10173,9.739290,9.965104,20.179010,73.716472,0.0,26.420632,1044.531967,363.330438,High Demand Zone,High order volume with strong density index.
2,2,4700,10.137983,9.931915,26.622653,76.283215,0.0,12.206524,463.603064,868.932001,Emerging Growth Zone,Mid-demand zone with growth potential.
3,0,4283,10.639384,9.928788,18.336138,76.065947,0.0,11.123520,402.560883,102.908395,Central Delivery Zone,Cluster centroid is near network center; good ...
4,4,4029,7.295446,10.042194,18.346399,76.066398,0.0,10.463848,552.262362,102.251226,Central Delivery Zone,Cluster centroid is near network center; good ...
5,5,3792,9.574792,9.978903,23.852422,84.800064,0.0,9.848327,396.039929,980.414390,Low Activity Zone,Low demand volume relative to other zones.


,cluster,zone_label,zone_reason
0,4,Central Delivery Zone,Cluster centroid is near network center; good ...
1,1,High Demand Zone,High order volume with strong density index.
2,4,Central Delivery Zone,Cluster centroid is near network center; good ...
3,1,High Demand Zone,High order volume with strong density index.
4,1,High Demand Zone,High order volume with strong density index.


ZoneLabelRuleSummary(high_demand_threshold=8804.75, low_activity_threshold=4092.5, high_density_threshold=675.3297635825357, centrality_threshold_km=298.2249272687038, southern_lat_threshold=18.33870331834506)
